# V7.2 — Corrected Qwen parser, offline checkpoint replay (NO GPU)

Khôi phục kết quả của 27 truy vấn đã chạy tiling dựa trên **raw_full**, **raw_crops** trong ZIP kết quả. Quy tắc gốc: đủ ≥4 detection, Qwen ảnh đầy đủ YES, ít nhất 4/6 crop YES và đủ dòng parse.

Không suy luận lại CountGD++ hoặc Qwen; không sử dụng ground truth trong logic chọn cặp/gate. Ground truth chỉ dùng để tính metrics sau khi đã cố định dự đoán.

Add Input ZIP `V7_2_Adaptive_Tiling_Exploratory_1301.zip` trên Kaggle. Kaggle có thể tự giải nén ZIP; notebook hỗ trợ cả hai.

In [ ]:
from pathlib import Path
import re, io, zipfile, numpy as np, pandas as pd
from IPython.display import display, FileLink
ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT=WORK/'v72_parser_fixed'; OUT.mkdir(exist_ok=True)
ZIP_NAME='V7_2_Adaptive_Tiling_Exploratory_1301.zip'
need=['v72_exploratory_1301_predictions.csv','v72_visual_gate_checkpoint.csv']
zip_paths=list(ROOT.rglob(ZIP_NAME))
if zip_paths:
    z=zipfile.ZipFile(zip_paths[0])
    tables={x:pd.read_csv(io.BytesIO(z.read(x))) for x in need}
    print('Read ZIP:',zip_paths[0])
else:
    tables={}
    for name in need:
        paths=list(ROOT.rglob(name))
        if not paths:raise FileNotFoundError('Missing input '+name+'; add the V7.2 export to Kaggle Input')
        tables[name]=pd.read_csv(paths[0]);print('Read CSV:',paths[0])
data=tables[need[0]]; logs=tables[need[1]]
assert len(data)==1301 and data.group_id.is_unique
assert len(logs)==27 and logs.group_id.is_unique and logs.status.eq('ok').all()
print('Ready:',len(data),'pairs /',len(logs),'gate logs')

In [ ]:
# PARSER FIX: use actual word boundaries, not U+0008 backspace characters.
FULL_RE=re.compile(r'\b(?:YES|NO|UNCERTAIN)\b',re.I)
LINE_RE=re.compile(r'^\s*(?:ID\s*)?(\d+)\s*\|\s*(YES|NO)\s*$',re.I)
assert FULL_RE.findall('YES')==['YES']
def parse_full(text):
    s='' if pd.isna(text) else str(text)
    tokens=FULL_RE.findall(s.upper())
    return tokens[0].upper() if len(tokens)==1 else 'UNCERTAIN'
def parse_crops(text,n):
    parsed={}
    if not pd.isna(text):
        for line in str(text).splitlines():
            m=LINE_RE.fullmatch(line.strip())
            if m and 0<=int(m.group(1))<n:
                parsed[int(m.group(1))]=m.group(2).upper()
    return parsed
rows=[]
for r in logs.itertuples(index=False):
    full=parse_full(r.raw_full)
    n=int(r.n_crops)
    parsed=parse_crops(r.raw_crops,n) if n>0 else {}
    yes=sum(v=='YES' for v in parsed.values())
    confirm=(int(r.tile_count)>=4 and full=='YES' and n>=4 and len(parsed)==n and yes>=4)
    rows.append({'group_id':r.group_id,'image_name':r.image_name,'target':r.target,
      'tile_count':int(r.tile_count),'raw_full':r.raw_full,
      'full_presence_fixed':full,'crop_yes_from_logs':yes,'n_crops':n,
      'gate_old':r.gate,'gate_fixed':'CONFIRMED' if confirm else r.gate,
      'v72_fixed_count':int(r.tile_count if confirm else r.v7_count),'passed_gate':confirm})
gates=pd.DataFrame(rows)
print('Gate confirmed:',int(gates.passed_gate.sum()),'/',len(gates))
display(gates[['image_name','target','tile_count','full_presence_fixed','crop_yes_from_logs','gate_fixed']])

In [ ]:
# Evaluation stage: only HERE read gt_count/pair_type
all_new=data.merge(gates[['group_id','v72_fixed_count','gate_fixed']],on='group_id',how='left',validate='one_to_one')
all_new['v72_fixed_count']=all_new.v72_fixed_count.fillna(all_new.v7_count).astype(int)
all_new['v7_abs_error_recalc']=(all_new.v7_count-all_new.gt_count).abs()
all_new['v72_fixed_abs_error']=(all_new.v72_fixed_count-all_new.gt_count).abs()
all_new['delta_error']=all_new.v72_fixed_abs_error-all_new.v7_abs_error_recalc
all_new['changed']=all_new.v72_fixed_count.ne(all_new.v7_count)
metrics=[]
for subset,sub in [('POSITIVE',all_new[all_new.pair_type=='POSITIVE']),
                   ('HARD_NEGATIVE',all_new[all_new.pair_type=='HARD_NEGATIVE']),
                   ('ALL',all_new)]:
    for method,col in [('V7','v7_count'),('V7.2_fixed','v72_fixed_count')]:
        err=(sub[col]-sub.gt_count).abs()
        metrics.append({'subset':subset,'method':method,'n':len(sub),'MAE':err.mean(),
          'RMSE':np.sqrt(((sub[col]-sub.gt_count)**2).mean()),
          'FPR':(sub[col]>0).mean() if subset=='HARD_NEGATIVE' else np.nan,
          'positive_zero_rate':(sub[col]==0).mean() if subset=='POSITIVE' else np.nan})
metric_df=pd.DataFrame(metrics); changed=all_new[all_new.changed].sort_values('delta_error')
print(metric_df.to_string(index=False))
print('Changed:',len(changed),'total reduction in absolute error:',int(-changed.delta_error.sum()))
display(changed[['image_name','target','gt_count','v7_count','v72_fixed_count','delta_error']])
assert len(changed)==15 and changed.pair_type.eq('POSITIVE').all()
assert int(changed.delta_error.sum())==-1736
assert all_new.group_id.is_unique
for name,df in [('V7_2_reparsed_1301_predictions.csv',all_new),
 ('V7_2_reparsed_metrics.csv',metric_df),('V7_2_reparsed_changed_15.csv',changed),
 ('V7_2_reparsed_visual_gate_27.csv',gates)]:
    df.to_csv(OUT/name,index=False,encoding='utf-8-sig')
archive=WORK/'V7_2_Parser_Fixed_NoGPU_1301.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'):z.write(p,p.name)
print('Created:',archive)
display(FileLink(str(archive)))